In [31]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass, duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face Token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-01/*.parquet')"

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [32]:
# 1. Unit of Analysis (Grain): One row represents one content item (web page) for one client on one specific calendar day (report_date × client_hash_id × content_hash_id).

# 2. Time Window: Mid-panel month of March 2026 (2026-03-01 to 2026-03-31). I use a mid-panel month because the final month (June 2026) is reserved as a sealed test month.
print(
    """
1. Unit of Analysis (Grain): One row represents one content item (web page) for one client on one specific calendar day (report_date × client_hash_id × content_hash_id).

2. Time Window: Mid-panel month of March 2026 (2026-03-01 to 2026-03-31). I use a mid-panel month because the final month (June 2026) is reserved as a sealed test month.
    """
)


1. Unit of Analysis (Grain): One row represents one content item (web page) for one client on one specific calendar day (report_date × client_hash_id × content_hash_id).

2. Time Window: Mid-panel month of March 2026 (2026-03-01 to 2026-03-31). I use a mid-panel month because the final month (June 2026) is reserved as a sealed test month.
    


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [33]:
# Fields Classification:

# Features (Inputs): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, content_age_days. (All knowable before or at the moment of prediction).
# Label (Target): is_declining_label (1 if impressions drop by >20% in the future period, else 0).
# Context: content_hash_id, client_hash_id, report_date (Used ONLY for grouping, joining, and client-holdout splits. Never model features).
# Excluded: trend_direction, trend_pct, health_score.
# Why excluded? trend_direction and trend_pct are derived from the future label period (using them is feature leakage). health_score is a hand-written product flag, not a raw search signal.
print("""
Fields Classification:

1. Features (Inputs): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, content_age_days. (All knowable before or at the moment of prediction).
2. Label (Target): is_declining_label (1 if impressions drop by >20% in the future period, else 0).
3. Context: content_hash_id, client_hash_id, report_date (Used ONLY for grouping, joining, and client-holdout splits. Never model features).
4. Excluded: trend_direction, trend_pct, health_score.
5. Why excluded? trend_direction and trend_pct are derived from the future label period (using them is feature leakage). health_score is a hand-written product flag, not a raw search signal.
""")


Fields Classification:

1. Features (Inputs): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, content_age_days. (All knowable before or at the moment of prediction).
2. Label (Target): is_declining_label (1 if impressions drop by >20% in the future period, else 0).
3. Context: content_hash_id, client_hash_id, report_date (Used ONLY for grouping, joining, and client-holdout splits. Never model features).
4. Excluded: trend_direction, trend_pct, health_score.
5. Why excluded? trend_direction and trend_pct are derived from the future label period (using them is feature leakage). health_score is a hand-written product flag, not a raw search signal.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [34]:
# 1. Verification Query 1: Grain Check on March 2026
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c 
    FROM {FACT} 
    GROUP BY 1, 2, 3 
    HAVING c > 1
""").df()
print("Grain check (duplicates, expect 0):", len(q1))

# 2. Verification Query 2: Row Count and Date Span for March 2026
q2 = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as start_date, MAX(report_date) as end_date 
    FROM {FACT}
""").df()
print("\nRow count & dates for January 2026:")
print(q2)

# 3. Verification Query 3: Availability Check (client_has_ga4 IS TRUE)
q3 = con.sql(f"""
    SELECT COUNT(*) as total_rows, 
           COUNT(CASE WHEN client_has_ga4 IS TRUE THEN 1 END) as ga4_available_rows,
           ROUND(COUNT(CASE WHEN client_has_ga4 IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as pct_ga4_available
    FROM {FACT}
""").df()
print("\nAvailability check:")
print(q3)

# 4. Feature Frame & Leakage Trap
df = con.sql(f"""
    WITH split_month AS (
        SELECT content_hash_id,
               -- First half of March (Days 1-15): HONEST FEATURES (PAST)
               SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_impressions ELSE 0 END) AS feat_imp_first15,
               SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_clicks ELSE 0 END) AS feat_clk_first15,
               AVG(CASE WHEN DAY(report_date) <= 15 THEN gsc_avg_position ELSE NULL END) AS feat_pos_first15,
               SUM(CASE WHEN DAY(report_date) <= 15 THEN ga4_sessions ELSE 0 END) AS feat_sess_first15,
               
               -- Second half of March (Days 16-31): FUTURE OUTCOME (LABEL PERIOD)
               SUM(CASE WHEN DAY(report_date) > 15 THEN gsc_impressions ELSE 0 END) AS outcome_imp_second15
        FROM {FACT}
        GROUP BY content_hash_id
        HAVING feat_imp_first15 >= 50  -- Only evaluate pages with measurable baseline traffic
    )
    SELECT content_hash_id,
           feat_imp_first15,
           feat_clk_first15,
           COALESCE(feat_pos_first15, 50.0) AS feat_pos_first15,
           feat_sess_first15,
           
           -- LEAKED FEATURE: Peeking into second half impressions
           outcome_imp_second15 AS leaked_future_impressions,
           
           -- HONEST LABEL: Did impressions drop by >20% in the second half?
           CASE WHEN outcome_imp_second15 < (0.80 * feat_imp_first15) THEN 1 ELSE 0 END AS is_declining_label
    FROM split_month
""").df().dropna()

# Define Honest Features vs Leaked Features
X_honest = df[['feat_imp_first15', 'feat_clk_first15', 'feat_pos_first15', 'feat_sess_first15']]
X_leaked = df[['feat_imp_first15', 'feat_clk_first15', 'feat_pos_first15', 'feat_sess_first15', 'leaked_future_impressions']]
y = df['is_declining_label']

# Split data into train and test sets
X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.25, random_state=42, stratify=y)

# 1. Model WITH Leakage Trap
clf_leaked = RandomForestClassifier(random_state=42).fit(X_tr_l, y_tr)
score_leaked = clf_leaked.score(X_te_l, y_te)
print("\nScore WITH Leakage Trap (Cheating):", round(score_leaked, 4))

# 2. Model WITHOUT Leakage Trap (Honest)
clf_honest = RandomForestClassifier(random_state=42).fit(X_tr_h, y_tr)
score_honest = clf_honest.score(X_te_h, y_te)
print("Honest Score WITHOUT Leakage (Realistic):", round(score_honest, 4))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (duplicates, expect 0): 0

Row count & dates for January 2026:
   total_rows start_date   end_date
0     7890817 2026-01-01 2026-01-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability check:
   total_rows  ga4_available_rows  pct_ga4_available
0     7890817             1960542              24.85


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Score WITH Leakage Trap (Cheating): 0.9906
Honest Score WITHOUT Leakage (Realistic): 0.8355


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [35]:
# Limitation: Unbalanced panel history. Clients joined at different dates (gsc_data_start), and early rows have zero-filled GA4 data (ga4_data_available = FALSE). Filter by ga4_data_available IS TRUE when using session metrics.
print("""
Limitation: Unbalanced panel history. Clients joined at different dates (gsc_data_start), and early rows have zero-filled GA4 data (ga4_data_available = FALSE). Filter by ga4_data_available IS TRUE when using session metrics.
""")


Limitation: Unbalanced panel history. Clients joined at different dates (gsc_data_start), and early rows have zero-filled GA4 data (ga4_data_available = FALSE). Filter by ga4_data_available IS TRUE when using session metrics.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.